### Environment setup

conda config --add channels conda-forge

conda create -n rag_chatbot2 python pandas nltk numpy jq "langchain<1.0" langchain-community chromadb langchain-chroma langchain-ollama langchain-openai streamlit

conda activate rag_chatbot2

pip install grobid_tei_xml rank_bm25

In [ ]:
import json
from typing import Iterator, List, Optional
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import OllamaEmbeddings
from langchain_chroma import Chroma

import pandas as pd
from langchain.chains import RetrievalQA

from langchain_ollama import OllamaLLM
import os
from langchain_core.documents import Document
import streamlit as st
import grobid_tei_xml
import re
from pprint import pprint
from tqdm import tqdm

## Read chunks. Combine into one jsonl file and update "id" field

In [ ]:
def read_jsonl_to_documents(data_path, tag):
    docs= []
    with open(data_path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as e:
                # skip/notify on malformed line
                print(f"Skipping line {i+1}: JSON decode error: {e}")
                continue

            # Extract page_content and metadata, with safe fallbacks
            page_content = obj.get("page_content", "") or ""
            metadata = obj.get("metadata") or {}
            metadata["tag"] = f"{tag}_{metadata["chunk_id"]}"  # Add the tag to metadata

            docs.append(Document(page_content=page_content, metadata=metadata))
    return docs

In [ ]:
import json
from datetime import datetime, date
from langchain_core.documents import Document  # or: from langchain.schema import Document

def json_safe(obj):
    # Recursively make metadata JSON-serializable
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    if isinstance(obj, (list, tuple)):
        return [json_safe(x) for x in obj]
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (datetime, date)):
        return obj.isoformat()
    # Fallback: string representation
    return str(obj)

def save_docs_json(docs, ques_id):
    serializable = []
    for rank, d in enumerate(docs, start=1):
        # Optionally keep retrieval info
        md = dict(d.metadata or {})
        md.setdefault("retrieval_rank", rank)
        md.setdefault("ques_id", ques_id)
        serializable.append({
        "page_content": d.page_content,
        "metadata": json_safe(md),
        })
    return serializable

## Create vector database

In [ ]:
# Qwen3 has a context window of 40K tokens and is a 8B-paramter model
embeddings = OllamaEmbeddings(model="qwen3-embedding:latest")
vector_store = Chroma(
    collection_name="periop_pain_guidelines_collection",
    embedding_function=embeddings,
    persist_directory= "periop_pain_chroma_dbs/"
    )

## Extract the top-k similar chunks for the given patient vignette and query

In [ ]:
llm = OllamaLLM(model= "deepseek-r1:8b")
    # Create Retriever
 
retriever_similarity= vector_store.as_retriever(
    search_type= "similarity", # mmr is an alternative
    search_kwargs= {"k": 10})

retriever_mmr= vector_store.as_retriever(
    search_type= "mmr", # mmr is an alternative
    search_kwargs= {"k": 10})

rag= RetrievalQA.from_chain_type(llm = OllamaLLM(model= "deepseek-r1:8b"), retriever= retriever_similarity, return_source_documents=True)
openthinker_rag= RetrievalQA.from_chain_type(llm = OllamaLLM(model= "openthinker:7b"), retriever= retriever_similarity, return_source_documents=True)

rag_mmr= RetrievalQA.from_chain_type(llm = OllamaLLM(model= "deepseek-r1:8b"), retriever= retriever_mmr, return_source_documents=True)
openthinker_rag_mmr= RetrievalQA.from_chain_type(llm = OllamaLLM(model= "openthinker:7b"), retriever= retriever_mmr, return_source_documents=True)

In [ ]:
ques_txt = "According to perioperative pain management guidelines, what are the recommended strategies for managing acute postoperative pain in patients with a history of opioid use disorder?"

## Reading from the Q-Pain dataset, extracting the medical vignettes and retrieving the top-10 chunks based on qwen-3 embeddings similarity and MMR

In [ ]:
qpain_postop_data = pd.read_csv("../../data/data_post_op.csv")
qpain_postop_data.head()

In [ ]:
qpain_postop_data.shape

In [ ]:
# define function that take as input a list of Documents and returns one string to concatenate the page contents of the retrieved documents, along with their metadata tags for reference
def concatenate_docs(docs):
    concatenated = ""
    for doc in docs:
        tag = doc.metadata.get("tag", "no_tag")
        concatenated += f"[{tag}] {doc.page_content}\n\n"
    return concatenated.strip()

In [ ]:
retrieved_similar_evidence_list, retrieved_mmr_evidence_list = [], []

for row_id in range(len(qpain_postop_data)):
    ques_txt= "Vignette: " + qpain_postop_data.iloc[row_id, 0] +".\nQuestion: " + qpain_postop_data.iloc[row_id, 1]
    embed_similar = concatenate_docs(rag.invoke(input= ques_txt)['source_documents']) # returns a list of Document objects with page_content and metadata
    embed_mmr = concatenate_docs(rag_mmr.invoke(input= ques_txt)['source_documents']) # returns a list of Document objects with page_content and metadata

    retrieved_similar_evidence_list.append(embed_similar)
    retrieved_mmr_evidence_list.append(embed_mmr)
    print(f"Processed row id {row_id}")

In [ ]:
qpain_postop_data1 = qpain_postop_data.assign(retrieved_similar_evidence= retrieved_similar_evidence_list, retrieved_mmr_evidence= retrieved_mmr_evidence_list)

In [ ]:
qpain_postop_data1.head()

In [ ]:
## Exporting the dataset
qpain_postop_data1.to_csv("../../data/qpain_postop_data_with_retrieved_evidence.csv", index=False)

In [ ]:
'''
print("Start creating the vector store")
embedding_batch_size = 50

for i in tqdm(range(0, len(composite_list_of_chunks), embedding_batch_size), desc="Adding documents to vector store"):
    batch = composite_list_of_chunks[i:i+embedding_batch_size]
    batch_tags = [doc.metadata["tag"] for doc in batch]
    #print(f"Adding batch {i//embedding_batch_size + 1} with tags: {batch_tags}")
    vector_store.add_documents(documents=batch, ids=batch_tags)
'''